1. Setup + ENV

In [3]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Notebook location: api/notebooks
API_ROOT = Path("..").resolve()        # .../ai-tv-pro/api
DATA_DIR = (API_ROOT / "data").resolve()

# Load env
load_dotenv(API_ROOT / ".env")

# Allow imports from api/
sys.path.insert(0, str(API_ROOT))

print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))

OPENAI_API_KEY present: True


2. Personas - TV use-case

In [4]:
from ragas.testset.persona import Persona

personas = [
    Persona(
        name="IPTV Product Manager",
        role_description=(
            "Focuses on monthly trends, top channels, and content type split "
            "to inform product and partnership decisions."
        ),
    ),
    Persona(
        name="Content Strategy Analyst",
        role_description=(
            "Analyzes which channels and playback types drive engagement. "
            "Asks ranking, comparison, and share-of-viewing questions."
        ),
    ),
    Persona(
        name="Network Ops Analyst",
        role_description=(
            "Investigates anomalies and sudden changes in viewing minutes or viewers. "
            "Needs precise numbers and month-over-month comparisons."
        ),
    ),
    Persona(
        name="Customer Care Lead",
        role_description=(
            "Needs quick answers to explain user complaints and service issues, "
            "especially around live events vs replay viewing patterns."
        ),
    ),
]

print("Personas:", len(personas))
for p in personas:
    print("-", p.name)

c:\Users\gpudja\OneDrive - Hrvatski Telekom\00_Posao\RAZNO\99_EDUKACIJE\16_AI_Eng_Bootcamp\certification_challenge\ai-tv-pro\api\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\gpudja\OneDrive - Hrvatski Telekom\00_Posao\RAZNO\99_EDUKACIJE\16_AI_Eng_Bootcamp\certification_challenge\ai-tv-pro\api\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Personas: 4
- IPTV Product Manager
- Content Strategy Analyst
- Network Ops Analyst
- Customer Care Lead


3. CSV file

In [5]:
from pathlib import Path
import pandas as pd

# Notebook is in api/notebooks
API_ROOT = Path("..").resolve()  # -> api
csv_path = API_ROOT / "tv_data" / "Monthly iptv - channel monthly rating.csv"

print("csv_path:", csv_path)
print("exists:", csv_path.exists())

# Load with semicolon delimiter
df = pd.read_csv(csv_path, sep=";")

print("rows:", len(df), "cols:", len(df.columns))
print("columns:", list(df.columns))

df.head(3)

csv_path: C:\Users\gpudja\OneDrive - Hrvatski Telekom\00_Posao\RAZNO\99_EDUKACIJE\16_AI_Eng_Bootcamp\certification_challenge\ai-tv-pro\api\tv_data\Monthly iptv - channel monthly rating.csv
exists: True
rows: 138642 cols: 6
columns: ['month_partition', 'watch_date', 'channelname', 'content_playback_type', 'daily_total_minute', 'nr_unique_viewers']


,month_partition,watch_date,channelname,content_playback_type,daily_total_minute,nr_unique_viewers
0,202601,2026-01-01,BBC Earth,LTV,"163974,13",4413
1,202601,2026-01-01,Brazzers TV,LTV,"29304,38",3768
2,202601,2026-01-01,Das Erste,LTV,"26797,96",509


4. Text from CSV

In [11]:
from langchain_core.documents import Document
import pandas as pd

# Column names (confirmed from dataset)
COL_MONTH = "month_partition"
COL_DATE = "watch_date"
COL_CHANNEL = "channelname"
COL_PLAYBACK = "content_playback_type"
COL_MINUTES = "daily_total_minute"
COL_VIEWERS = "nr_unique_viewers"

# ✅ Speed tuning: fewer rows per month -> fewer tokens -> faster testset generation
TOP_N = 20

# ---- 1. Basic type cleanup ----
df[COL_DATE] = pd.to_datetime(df[COL_DATE], errors="coerce")
df[COL_MINUTES] = pd.to_numeric(df[COL_MINUTES], errors="coerce")
df[COL_VIEWERS] = pd.to_numeric(df[COL_VIEWERS], errors="coerce")

print("After type conversion:")
print(df.dtypes)

# ---- 2. Aggregate to monthly level per channel & playback ----
agg = (
    df.groupby([COL_MONTH, COL_CHANNEL, COL_PLAYBACK], dropna=False)
      .agg(
          total_minutes=(COL_MINUTES, "sum"),
          unique_viewers=(COL_VIEWERS, "sum"),
          days=(COL_DATE, "nunique"),
      )
      .reset_index()
)

print("Aggregated rows:", len(agg))

# ---- 3. Convert each month into a text document ----
def month_doc(month_df: pd.DataFrame, month_id: str, top_n: int = TOP_N) -> str:
    month_df = month_df.sort_values("total_minutes", ascending=False).head(top_n)

    lines = []
    lines.append(f"IPTV Monthly Report for month_partition={month_id}")
    lines.append("Fields: channelname, content_playback_type, total_minutes, unique_viewers, days")

    # small summary to help LLM quickly anchor
    if len(month_df) > 0:
        top = month_df.iloc[0]
        lines.append(
            f"Summary: Top entry is {top[COL_CHANNEL]} ({top[COL_PLAYBACK]}) "
            f"with {int(top['total_minutes'])} minutes and {int(top['unique_viewers'])} viewers."
        )

    lines.append(f"Top {top_n} entries:")

    for _, r in month_df.iterrows():
        lines.append(
            f"- channel={r[COL_CHANNEL]} | playback={r[COL_PLAYBACK]} "
            f"| minutes={int(r['total_minutes']) if pd.notna(r['total_minutes']) else 'NA'} "
            f"| viewers={int(r['unique_viewers']) if pd.notna(r['unique_viewers']) else 'NA'} "
            f"| days={int(r['days']) if pd.notna(r['days']) else 'NA'}"
        )

    return "\n".join(lines)

docs = []
for month_id, g in agg.groupby(COL_MONTH):
    docs.append(
        Document(
            page_content=month_doc(g, str(month_id), top_n=TOP_N),
            metadata={"month_partition": str(month_id)}
        )
    )

print("Documents created:", len(docs))
print("TOP_N used:", TOP_N)
print("\n--- Sample document preview ---\n")
print(docs[0].page_content[:800])

After type conversion:
month_partition                   int64
watch_date               datetime64[ns]
channelname                      object
content_playback_type            object
daily_total_minute              float64
nr_unique_viewers                 int64
dtype: object
Aggregated rows: 5339
Documents created: 14
TOP_N used: 20

--- Sample document preview ---

IPTV Monthly Report for month_partition=202412
Fields: channelname, content_playback_type, total_minutes, unique_viewers, days
Summary: Top entry is HRT 1 (LTV) with 33726930 minutes and 4972636 viewers.
Top 20 entries:
- channel=HRT 1 | playback=LTV | minutes=33726930 | viewers=4972636 | days=31
- channel=Arenasport 1 | playback=LTV | minutes=6922661 | viewers=1131978 | days=31
- channel=Doma TV | playback=LTV | minutes=6326153 | viewers=1890390 | days=31
- channel=N1 | playback=LTV | minutes=3939420 | viewers=1687682 | days=31
- channel=HRT 4 | playback=LTV | minutes=3754198 | viewers=1850634 | days=31
- channel=HRT 3 | 

5. Testset generator + personas

In [16]:
import os
from openai import OpenAI

from ragas.testset import TestsetGenerator
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings

import ragas

MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

client = OpenAI()

llm = llm_factory(MODEL, client=client)

embeddings = OpenAIEmbeddings(
    client=client,
    model=EMBED_MODEL
)

generator = TestsetGenerator(
    llm=llm,
    embedding_model=embeddings,
    persona_list=personas
)

print("Generator ready:", MODEL)
print("Embedding model:", EMBED_MODEL)
print("ragas version:", ragas.__version__)

Generator ready: gpt-4o-mini
Embedding model: text-embedding-3-small
ragas version: 0.4.3


6. Generate testset

In [ ]:
docs_for_gen = docs  

testset = generator.generate_with_langchain_docs(
    docs_for_gen,
    testset_size=25
)

df_test = testset.to_pandas()
print("Generated rows:", len(df_test))
print("Columns:", list(df_test.columns))
df_test.head(10)

Applying HeadlinesExtractor:   0%|          | 0/14 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/14 [00:00<?, ?it/s]c:\Users\gpudja\OneDrive - Hrvatski Telekom\00_Posao\RAZNO\99_EDUKACIJE\16_AI_Eng_Bootcamp\certification_challenge\ai-tv-pro\api\.venv\Lib\site-packages\ragas\testset\transforms\base.py:198: UserWarning: Using sync embedding model OpenAIEmbeddings in async context. This may impact performance. Consider using an async-compatible embedding model for better performance.
  property_name, property_value = await self.extract(node)
Generating Samples: 100%|██████████| 27/27 [01:47<00:00,  3.97s/it]


Generated rows: 27
Columns: ['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name']


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,In the context of the IPTV Monthly Report for ...,[IPTV Monthly Report for month_partition=20241...,"In the IPTV Monthly Report for December 2024, ...",Customer Care Lead,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
1,What are the viewer statistics for Doma TV ove...,[- channel=HRT 1 | playback=LTV | minutes=3372...,"Doma TV had a total of 6,326,153 minutes of pl...",Customer Care Lead,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
2,What top channel in IPTV Monthly Report?,[IPTV Monthly Report for month_partition=20250...,Top entry is Nova TV (LTV) with 51622038 minut...,Customer Care Lead,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
3,How does RTL perform in terms of viewer engage...,[Top 20 entries:\n- channel=Nova TV | playback...,"RTL has a total of 4,999,005 viewers and 17,99...",IPTV Product Manager,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
4,What is the LTV and how many minutes and viewe...,[IPTV Monthly Report for month_partition=20250...,The LTV for MAXSport 1 is 14939709 minutes and...,Customer Care Lead,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
5,How many viewing minutes and viewers did STAR ...,[Top 20 entries:\n- channel=MAXSport 1 | playb...,"In the last 28 days, STAR Movies had 1,491,679...",Network Ops Analyst,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
6,Wht are the total minits and unique viwers for...,[IPTV Monthly Report for month_partition=20250...,"The total minutes for HRT 1 are 26,220,170 and...",Network Ops Analyst,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
7,How does the viewership of the STAR Crime chan...,[Top 20 entries:\n- channel=HRT 1 | playback=L...,"The STAR Crime channel had a total of 838,027 ...",Customer Care Lead,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
8,What top channel in IPTV report have most minu...,[IPTV Monthly Report for month_partition=20250...,Top entry is RTL (LTV) with 17311677 minutes a...,Network Ops Analyst,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
9,What are the top channel statistics for RTL in...,[<1-hop>\n\nIPTV Monthly Report for month_part...,"In the IPTV Monthly Report for January 2026, t...",NaN,NaN,NaN,multi_hop_abstract_query_synthesizer


7. Save TV test data

In [19]:
from pathlib import Path

out_path = Path("../tv_data/testset_tv.json")

df_test.to_json(
    out_path,
    orient="records",
    indent=2
)

print(f"Saved testset to {out_path.resolve()}")
print("Rows saved:", len(df_test))

Saved testset to C:\Users\gpudja\OneDrive - Hrvatski Telekom\00_Posao\RAZNO\99_EDUKACIJE\16_AI_Eng_Bootcamp\certification_challenge\ai-tv-pro\api\tv_data\testset_tv.json
Rows saved: 27
